<a href="https://colab.research.google.com/github/NatSy77/mlops_ai-projet6et8/blob/main/06_final_model_export.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Initiez-vous au MLOps (partie 1/2)
Final model export

Ce notebook finalise le modèle retenu pour le projet de credit scoring :

- recharge les données complètes d'entraînement
- applique le même encodage que dans les notebooks précédents
- réentraîne le modèle final retenu sur tout le dataset
- sauvegarde les artefacts nécessaires pour la suite API
- logge le modèle final dans MLflow


In [2]:
!pip install -q mlflow lightgbm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 14.6 MB/s eta 0:00:00


In [3]:
import os
import re
import json
import numpy as np
import pandas as pd

import mlflow
import mlflow.lightgbm

from lightgbm import LGBMClassifier


## Configuration

In [4]:
RANDOM_STATE = 42

MLFLOW_TRACKING_URI = "file:///content/drive/MyDrive/Colab Notebooks/AI_projet_6&8/mlruns"
EXPERIMENT_NAME = "credit_scoring_final_model"

ARTIFACT_DIR = "/content/final_model_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)


## Configuration MLflow

In [5]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.end_run()

exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if exp is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
    print("Nouvelle expérience créée :", experiment_id)
else:
    print("Expérience existante :", exp.experiment_id)

mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", mlflow.get_experiment_by_name(EXPERIMENT_NAME))


Nouvelle expérience créée : 830388907310039175
Tracking URI: file:///content/drive/MyDrive/Colab Notebooks/AI_projet_6&8/mlruns
Experiment: <Experiment: artifact_location=('file:///content/drive/MyDrive/Colab '
 'Notebooks/AI_projet_6&8/mlruns/830388907310039175'), creation_time=1775459065434, experiment_id='830388907310039175', last_update_time=1775459065434, lifecycle_stage='active', name='credit_scoring_final_model', tags={}, workspace='default'>


/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


## Chargement des données complètes

In [7]:
from google.colab import drive

# Unmount Google Drive if already mounted to avoid conflicts
if os.path.exists('/content/drive'):
  print('Unmounting existing /content/drive...')
  try:
    drive.flush_and_unmount()
  except Exception as e:
    print(f'Error during unmount: {e}')

# Ensure the mount point is empty or doesn't exist
if os.path.exists('/content/drive'):
    if os.listdir('/content/drive'):
        print("Mount point '/content/drive' is not empty. Removing contents...")
        !rm -rf /content/drive/*

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

Unmounting existing /content/drive...
Drive not mounted, so nothing to flush and unmount.
Mount point '/content/drive' is not empty. Removing contents...
Mounted at /content/drive


In [9]:
X = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/AI_projet_6&8/credit_scoring_data/processed/X_train.csv")
y = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/AI_projet_6&8/credit_scoring_data/processed/y_train.csv").values.ravel()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positive rate:", y.mean())


X shape: (307511, 377)
y shape: (307511,)
Positive rate: 0.08072881945686496


## Encodage identique aux notebooks 04/05

In [10]:
def clean_feature_names(columns):
    cleaned = []
    seen = {}

    for col in columns:
        new_col = str(col)
        new_col = re.sub(r"[^A-Za-z0-9_]+", "_", new_col)
        new_col = re.sub(r"_+", "_", new_col)
        new_col = new_col.strip("_")

        if new_col == "":
            new_col = "feature"

        if new_col in seen:
            seen[new_col] += 1
            new_col = f"{new_col}_{seen[new_col]}"
        else:
            seen[new_col] = 0

        cleaned.append(new_col)

    return cleaned


In [11]:
X_encoded = pd.get_dummies(X, drop_first=False)
X_encoded.columns = clean_feature_names(X_encoded.columns)

print("Encoded shape:", X_encoded.shape)
print("Exemple de colonnes :", X_encoded.columns[:10].tolist())


Encoded shape: (307511, 501)
Exemple de colonnes : ['SK_ID_CURR', 'PREV_PRODUCT_COMBINATION_CASH_MEAN', 'PREV_NAME_PORTFOLIO_CARS_MEAN', 'DAYS_LAST_PHONE_CHANGE', 'PREV_NAME_GOODS_CATEGORY_MOBILE_MEAN', 'PREV_NAME_CONTRACT_STATUS_APPROVED_MEAN', 'BURO_BB_MONTHS_BALANCE_MAX_MEAN', 'PREV_CODE_REJECT_REASON_SYSTEM_MEAN', 'PREV_NAME_TYPE_SUITE_CHILDREN_MEAN', 'PREV_AMT_DOWN_PAYMENT_MEAN']


## Sauvegarde des colonnes encodées

In [12]:
encoded_columns = X_encoded.columns.tolist()

encoded_columns_path = os.path.join(ARTIFACT_DIR, "encoded_columns.json")
with open(encoded_columns_path, "w") as f:
    json.dump(encoded_columns, f)

print("encoded_columns.json saved")


encoded_columns.json saved


## Recalcul du poids de classe

In [13]:
n_negative = int((y == 0).sum())
n_positive = int((y == 1).sum())

scale_pos_weight = n_negative / n_positive
print("n_negative:", n_negative)
print("n_positive:", n_positive)
print("scale_pos_weight:", scale_pos_weight)


n_negative: 282686
n_positive: 24825
scale_pos_weight: 11.387150050352467


## Définition du modèle final retenu

In [14]:
final_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    force_col_wise=True
)

final_model


LGBMClassifier(colsample_bytree=0.8, force_col_wise=True, learning_rate=0.05,
               n_estimators=300, n_jobs=-1, objective='binary', random_state=42,
               scale_pos_weight=11.387150050352467, subsample=0.8)

## Entraînement final sur tout le dataset

In [15]:
final_model.fit(X_encoded, y)
print("Final model trained on full training dataset.")


[LightGBM] [Info] Number of positive: 24825, number of negative: 282686
[LightGBM] [Info] Total Bins 46778
[LightGBM] [Info] Number of data points in the train set: 307511, number of used features: 492
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486
Final model trained on full training dataset.


## Sauvegarde du seuil métier

In [16]:
threshold_data = {
    "threshold": 0.52,
    "fn_cost": 10,
    "fp_cost": 1
}

threshold_path = os.path.join(ARTIFACT_DIR, "threshold.json")
with open(threshold_path, "w") as f:
    json.dump(threshold_data, f)

print("threshold.json saved")
threshold_data


threshold.json saved


{'threshold': 0.52, 'fn_cost': 10, 'fp_cost': 1}

## Sauvegarde des métadonnées

In [17]:
metadata = {
    "model_name": "lightgbm_no_early_stopping",
    "dataset": "home_credit_final_train",
    "n_rows": int(X_encoded.shape[0]),
    "n_features": int(X_encoded.shape[1]),
    "threshold": 0.52,
    "fn_cost": 10,
    "fp_cost": 1,
    "random_state": RANDOM_STATE
}

metadata_path = os.path.join(ARTIFACT_DIR, "model_metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f)

print("model_metadata.json saved")
metadata


model_metadata.json saved


{'model_name': 'lightgbm_no_early_stopping',
 'dataset': 'home_credit_final_train',
 'n_rows': 307511,
 'n_features': 501,
 'threshold': 0.52,
 'fn_cost': 10,
 'fp_cost': 1,
 'random_state': 42}

## Logging MLflow du modèle final

In [19]:
import mlflow

MLFLOW_TRACKING_URI = "file:///content/drive/MyDrive/Colab Notebooks/AI_projet_6&8/mlruns"
EXPERIMENT_NAME = "credit_scoring_lightgbm_tuning"

# Reconnexion propre
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Fermer un éventuel run resté ouvert
mlflow.end_run()

# Recréer ou récupérer l'expérience par son nom
exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if exp is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
    print("Nouvelle expérience créée :", experiment_id)
else:
    experiment_id = exp.experiment_id
    print("Expérience existante :", experiment_id)

mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Expérience active:", mlflow.get_experiment_by_name(EXPERIMENT_NAME))

Expérience existante : 562923313514317022
Tracking URI: file:///content/drive/MyDrive/Colab Notebooks/AI_projet_6&8/mlruns
Expérience active: <Experiment: artifact_location=('file:///content/drive/MyDrive/Colab '
 'Notebooks/AI_projet_6&8/mlruns/562923313514317022'), creation_time=1775457794605, experiment_id='562923313514317022', last_update_time=1775457794605, lifecycle_stage='active', name='credit_scoring_lightgbm_tuning', tags={}, workspace='default'>


In [20]:
with mlflow.start_run(run_name="lightgbm_final_model"):
    mlflow.set_tag("project", "home_credit_scoring")
    mlflow.set_tag("stage", "finalization")
    mlflow.set_tag("model_type", "lightgbm")
    mlflow.set_tag("run_role", "final_model")
    mlflow.set_tag("goal", "business_cost_optimization")

    mlflow.log_param("model", "lightgbm")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("num_leaves", 31)
    mlflow.log_param("max_depth", -1)
    mlflow.log_param("min_child_samples", 20)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("objective", "binary")
    mlflow.log_param("scale_pos_weight", float(scale_pos_weight))
    mlflow.log_param("threshold", 0.52)
    mlflow.log_param("fn_cost", 10)
    mlflow.log_param("fp_cost", 1)
    mlflow.log_param("n_rows", int(X_encoded.shape[0]))
    mlflow.log_param("n_features", int(X_encoded.shape[1]))

    mlflow.log_artifact(encoded_columns_path)
    mlflow.log_artifact(threshold_path)
    mlflow.log_artifact(metadata_path)

    mlflow.lightgbm.log_model(final_model, name="final_model")

print("Final model logged to MLflow.")


2026/04/06 07:15:25 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Final model logged to MLflow.


## Vérification des artefacts

In [21]:
print("Artifacts directory content:")
print(os.listdir(ARTIFACT_DIR))


Artifacts directory content:
['encoded_columns.json', 'model_metadata.json', 'threshold.json']



## Notes pour la suite API

Pour l'API :

1. charger le modèle final
2. charger `encoded_columns.json`
3. charger `threshold.json`
4. appliquer `pd.get_dummies()` sur les nouvelles données
5. nettoyer les noms de colonnes avec la même fonction `clean_feature_names`
6. réaligner les colonnes avec :

```python
X_new = X_new.reindex(columns=encoded_columns, fill_value=0)
```

7. utiliser `predict_proba`
8. comparer la probabilité au seuil `0.52`

Note :
- X_test n'est pas utilisé ici afin d'éviter toute fuite de données.
- Il sera utilisé en phase d'inférence (API / dashboard).
